# House prices — held-out testing
This notebook loads the models saved by notebook 04 and compares their predictions with known SalePrice values matched by Id. The candidates were fitted on training rows and remain fixed during evaluation.

Run notebook 04 first, then run this notebook in order. Saved settings, data fingerprints and feature checks tie the scores to that training run. The report highlights the lowest held-out RMSE and flags large error gaps.


## Setup and matching feature definitions


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display, HTML

# Locate data from the notebook folder, project folder or repository root.
locations = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (candidate for location in locations
     for candidate in (location, location / "regression" / "house_price_prediction")
     if (candidate / "house_prices_dataset" / "train" / "train.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository or house_price_prediction folder.")
DATA_DIR = PROJECT_ROOT / "house_prices_dataset"
TRAIN_PATH = DATA_DIR / "train" / "train.csv"
GROUND_TRUTH_DIR = DATA_DIR / "ground_truth"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = PROJECT_ROOT / "reports"
TARGET = "SalePrice"
ID_COLUMN = "Id"
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_columns", 20)


In [ ]:
def engineer_features(X):
    """Apply row-wise domain rules; no statistics or labels are learned."""
    data = X.copy()
    if ID_COLUMN in data:
        data = data.drop(columns=[ID_COLUMN])

    # These categories use missing values to indicate an absent component.
    structural_missing = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu", "GarageCond", "GarageQual", "GarageType", "GarageFinish", "BsmtFinType1", "BsmtFinType2", "BsmtQual", "BsmtCond", "BsmtExposure"]
    for column in structural_missing:
        if column in data:
            data[column] = data[column].fillna("None")

    def add_sum(name, columns):
        existing = [c for c in columns if c in data.columns]
        if existing:
            data[name] = data[existing].fillna(0).sum(axis=1)

    add_sum("TotalSF", ["TotalBsmtSF", "1stFlrSF", "2ndFlrSF"])
    add_sum("TotalBathrooms", ["FullBath", "BsmtFullBath"])
    if "HalfBath" in data and "BsmtHalfBath" in data:
        data["TotalBathrooms"] = data["TotalBathrooms"] + 0.5 * data[["HalfBath", "BsmtHalfBath"]].fillna(0).sum(axis=1)
    add_sum("TotalPorchSF", ["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch", "WoodDeckSF"])
    add_sum("TotalQualityScore", ["OverallQual", "OverallCond"])

    # Keep descriptive aliases for the derived features.
    add_sum("PorchArea", ["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"])
    if "YrSold" in data and "YearBuilt" in data:
        data["Time_Held"] = (data["YrSold"] - data["YearBuilt"]).clip(lower=0)
        data["House_Age"] = data["Time_Held"]
    if "TotalBsmtSF" in data and "BsmtUnfSF" in data:
        data["Finished_BasementArea"] = (data["TotalBsmtSF"] - data["BsmtUnfSF"]).clip(lower=0)
    if "1stFlrSF" in data and "2ndFlrSF" in data and "GrLivArea" in data:
        # GrLivArea already includes above-ground floors: avoid double counting.
        data["Total_Living_Area"] = data["GrLivArea"]
    add_sum("Full_Baths", ["BsmtFullBath", "FullBath"])
    if "HalfBath" in data and "BsmtHalfBath" in data:
        data["Half_Baths"] = data[["BsmtHalfBath", "HalfBath"]].fillna(0).sum(axis=1)

    ordinal_maps = {
        "ExterQual": {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "Po": 0},
        "ExterCond": {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "Po": 0},
        "HeatingQC": {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "Po": 0},
        "KitchenQual": {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "Po": 0},
        "CentralAir": {"N": 0, "Y": 1},
        "BsmtQual": {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0},
        "BsmtCond": {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0},
        "FireplaceQu": {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0},
        "GarageQual": {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0},
        "GarageCond": {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0},
        "PoolQC": {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "None": 0},
        "BsmtExposure": {"Gd": 3, "Av": 2, "Mn": 1, "No": 0, "None": 0},
        "BsmtFinType1": {"GLQ": 5, "ALQ": 4, "BLQ": 3, "Rec": 2, "LwQ": 1, "Unf": 0.5, "None": 0},
        "BsmtFinType2": {"GLQ": 5, "ALQ": 4, "BLQ": 3, "Rec": 2, "LwQ": 1, "Unf": 0.5, "None": 0},
    }
    for column, mapping in ordinal_maps.items():
        if column in data:
            data[f"{column}_Ordinal"] = data[column].map(mapping)

    if "YrSold" in data and "YearBuilt" in data:
        data["AgeAtSale"] = (data["YrSold"] - data["YearBuilt"]).clip(lower=0)
    if "YrSold" in data and "YearRemodAdd" in data:
        data["YearsSinceRemodel"] = (data["YrSold"] - data["YearRemodAdd"]).clip(lower=0)
    if "GarageYrBlt" in data and "YrSold" in data:
        data["GarageAgeAtSale"] = (data["YrSold"] - data["GarageYrBlt"]).clip(lower=0)

    for source, name in (
        ("GarageArea", "HasGarage"),
        ("TotalBsmtSF", "HasBasement"),
        ("Fireplaces", "HasFireplace"),
        ("PoolArea", "HasPool"),
        ("2ndFlrSF", "HasSecondFloor"),
    ):
        if source in data:
            data[name] = (data[source].fillna(0) > 0).astype(int)

    if "OverallQual" in data and "GrLivArea" in data:
        data["QualityLivableArea"] = data["OverallQual"] * data["GrLivArea"]
    if "OverallQual" in data and "TotalSF" in data:
        data["QualityTotalSF"] = data["OverallQual"] * data["TotalSF"]
    if "ExterQual_Ordinal" in data and "ExterCond_Ordinal" in data:
        data["Exterior_Quality"] = data["ExterQual_Ordinal"] + data["ExterCond_Ordinal"]
    if "GarageQual_Ordinal" in data and "GarageCond_Ordinal" in data:
        data["Garage_Quality"] = data["GarageQual_Ordinal"] + data["GarageCond_Ordinal"]
    if "OverallQual" in data and "OverallCond" in data:
        data["Average_Quality"] = data["OverallQual"] + data["OverallCond"]
    # Use one canonical representation for exact aliases.
    data = data.drop(columns=["Time_Held", "House_Age", "Total_Living_Area", "TotalQualityScore"], errors="ignore")
    if "MSSubClass" in data:
        data["MSSubClass"] = data["MSSubClass"].astype(str)
    return data


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def metrics(actual, predicted):
    predicted = np.maximum(0, np.asarray(predicted))
    if not np.isfinite(predicted).all():
        raise ValueError("Nonfinite model predictions")
    return {"mae": float(mean_absolute_error(actual, predicted)),
            "rmse": float(np.sqrt(mean_squared_error(actual, predicted))),
            "rmsle": float(np.sqrt(mean_squared_error(np.log1p(actual), np.log1p(predicted)))),
            "r2": float(r2_score(actual, predicted))}


## Load and check the latest training run
The saved data and model fingerprints prevent evaluation of mismatched runs. No model is retrained here.


In [ ]:
import hashlib
import inspect
import json
import joblib

required = ["run_metadata.json", "candidates.joblib", "model_comparison.csv"]
missing = [name for name in required if not (ARTIFACTS_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Run 04_training.ipynb first. Missing artifacts: {missing}")
metadata = json.loads((ARTIFACTS_DIR / "run_metadata.json").read_text())
if metadata.get("format_version") != 1:
    raise ValueError("Unsupported saved run. Rerun 04_training.ipynb.")
for relative, expected in metadata["file_sha256"].items():
    path = PROJECT_ROOT / relative
    if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError(f"Data or training artifact changed: {relative}. Rerun 04_training.ipynb.")
signature = hashlib.sha256(inspect.getsource(engineer_features).encode()).hexdigest()
if signature != metadata["feature_signature"]:
    raise ValueError("Feature definitions differ from training. Keep the feature cells in 03, 04 and 05 in sync and rerun training.")
config = metadata["config"]
display(pd.Series(config, name="saved_training_configuration").to_frame())
X_raw = pd.read_csv(GROUND_TRUTH_DIR / "ground_truth_features.csv")
ground = pd.read_csv(GROUND_TRUTH_DIR / "ground_truth.csv")
if not X_raw[ID_COLUMN].equals(ground[ID_COLUMN]) or ground[ID_COLUMN].duplicated().any():
    raise ValueError("Ground-truth identifiers do not match unique prediction rows. Rerun training.")
X = engineer_features(X_raw)
candidates = joblib.load(ARTIFACTS_DIR / "candidates.joblib")
if list(candidates) != metadata["models"]:
    raise ValueError("Saved models do not match run metadata. Rerun training.")
training_comparison = pd.read_csv(ARTIFACTS_DIR / "model_comparison.csv")
if set(training_comparison.model) != set(candidates):
    raise ValueError("Training scores do not match saved models. Rerun training.")
print("Loaded candidates:", ", ".join(candidates))


## Reporting functions


In [ ]:
from html import escape
import plotly.express as px
import plotly.graph_objects as go
from plotly.io import to_html


def _table(df):
    return df.to_html(index=False, classes="dataframe", border=0, float_format=lambda x: f"{x:,.4f}")


def write_html(path: Path, title: str, intro: str, tables=None, figures=None, notes=None):
    tables = tables or []
    figures = figures or []
    notes = notes or []
    body = [f"<h1>{escape(title)}</h1><p>{escape(intro)}</p>"]
    for heading, table in tables:
        body.append(f"<h2>{escape(heading)}</h2>{_table(table)}")
    for index, (heading, figure) in enumerate(figures):
        body.append(f"<h2>{escape(heading)}</h2>{to_html(figure, full_html=False, include_plotlyjs=True if index == 0 else False)}")
    if notes:
        body.append("<h2>Evaluation notes</h2><ul>" + "".join(f"<li>{escape(n)}</li>" for n in notes) + "</ul>")
    html = """<!doctype html><html><head><meta charset='utf-8'><title>{}</title>
    <style>body{{font-family:Arial,sans-serif;max-width:1200px;margin:2rem auto;padding:0 1rem;color:#243447}}h1{{color:#12355b}}h2{{margin-top:2rem}}.dataframe{{border-collapse:collapse;width:100%}}.dataframe th,.dataframe td{{border:1px solid #ddd;padding:7px;text-align:right}}.dataframe th{{background:#eaf2f8;text-align:center}}</style>
    </head><body>{}</body></html>""".format(escape(title), "".join(body))
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(html, encoding="utf-8")


def comparison_figure(results: pd.DataFrame):
    return px.bar(results.sort_values("cv_rmse"), x="cv_rmse", y="model", orientation="h", error_x="cv_rmse_std", title="Cross-validation RMSE on SalePrice scale")


def stage_comparison_figure(results: pd.DataFrame, metric="rmse"):
    columns = [f"{stage}_{metric}" for stage in ("train", "validation", "test") if f"{stage}_{metric}" in results]
    flags = [name for name in ("overfitting_flag", "best_model_flag", "underperforming_flag") if name in results]
    frame = results.melt(id_vars=["model", *flags], value_vars=columns, var_name="dataset", value_name=metric)
    frame["dataset"] = frame["dataset"].str.replace(f"_{metric}", "", regex=False).map({"train": "Training", "validation": "Validation", "test": "Ground-truth testing"})
    direction = "lower is better" if metric != "r2" else "higher is better"
    figure = px.bar(frame, x="model", y=metric, color="dataset", barmode="group", hover_data=flags,
                     title=f"{metric.upper()}: training vs testing ({direction})")
    figure.update_layout(xaxis_tickangle=-35)
    return figure


def actual_predicted_figure(y_true, y_pred, title):
    frame = pd.DataFrame({"actual": y_true, "predicted": y_pred})
    fig = px.scatter(frame, x="actual", y="predicted", title=title, opacity=0.7)
    limits = [min(frame.min()), max(frame.max())]
    fig.add_trace(go.Scatter(x=limits, y=limits, mode="lines", name="perfect prediction"))
    return fig


def residual_figure(y_true, y_pred):
    frame = pd.DataFrame({"predicted": y_pred, "residual": y_true - y_pred})
    fig = px.scatter(frame, x="predicted", y="residual", title="Holdout residuals", opacity=0.7)
    fig.add_hline(y=0)
    return fig


## Evaluate every selected candidate and write reports


In [ ]:
records = []
predictions = pd.DataFrame({"Id": ground["Id"], "actual": ground[TARGET]})
for name, model in candidates.items():
    predicted = np.maximum(0, model.predict(X))
    records.append({"model": name, **metrics(ground[TARGET], predicted)})
    predictions[name] = predicted
results = pd.DataFrame(records).sort_values("rmse").reset_index(drop=True)
minimum = results.rmse.min()
results["best_model_flag"] = np.isclose(results.rmse, minimum, rtol=1e-9, atol=1e-9)
results["error_vs_best_ratio"] = results.rmse / max(minimum, 1e-9)
results["underperforming_flag"] = results.error_vs_best_ratio >= config["underperformance_ratio"]
best_names = results.loc[results.best_model_flag, "model"].tolist()
diagnostic_best = best_names[0]
results.to_csv(GROUND_TRUTH_DIR / "ground_truth_comparison.csv", index=False)
predictions.to_csv(GROUND_TRUTH_DIR / "ground_truth_predictions.csv", index=False)
combined = training_comparison.merge(
    results.rename(columns={name: f"test_{name}" for name in ("mae", "rmse", "rmsle", "r2")}), on="model", validate="one_to_one")
combined = combined.sort_values("test_rmse").reset_index(drop=True)
combined.to_csv(ARTIFACTS_DIR / "training_testing_comparison.csv", index=False)
metadata["best_observed_models"] = best_names
(ARTIFACTS_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2))
write_html(REPORTS_DIR / "testing_evaluation.html", "Ground-truth evaluation",
           "Every training-fitted candidate is compared against reserved known prices.",
           tables=[("Ground-truth ranking and flags", results)],
           figures=[("Actual vs predicted: lowest-RMSE candidate", actual_predicted_figure(ground[TARGET], predictions[diagnostic_best], diagnostic_best)),
                    ("Residuals: lowest-RMSE candidate", residual_figure(ground[TARGET], predictions[diagnostic_best]))],
           notes=["Known prices are joined by Id for scoring.", f"Best observed by test RMSE: {', '.join(best_names)}. The flag marks the lowest error in this comparison.",
                  f"Underperforming means test RMSE is at least {config['underperformance_ratio']} times the best observed RMSE."])
write_html(REPORTS_DIR / "model_comparison.html", "Model comparison",
           f"Best observed by ground-truth RMSE: {', '.join(best_names)}. The ranking summarizes this held-out split.",
           tables=[("Training and testing scores", combined)],
           figures=[("Dollar errors", stage_comparison_figure(combined)), ("Explained variance", stage_comparison_figure(combined, "r2"))],
           notes=["Training, validation and testing predictions use the same candidate fits; no refit on validation or ground truth.",
                  "Best-model and underperformance flags are relative to test RMSE; overfitting flags use validation/training RMSE.",
                  "The workflow ends at comparison. No candidate is selected, promoted or deployed."])
print(f"Best observed by ground-truth RMSE: {', '.join(best_names)}. No model selected.", flush=True)
comparison = results
display(comparison)
display(predictions.head(15))


## Results and error analysis
In the saved comparison, Linear Regression has the lowest held-out RMSE at $26,182 and R² of 0.911. Lasso is close at $26,469 RMSE and has lower MAE: $16,015 versus $16,823. Tuned XGBoost has the lowest MAE among these candidates at $15,193, but higher RMSE of $27,782. The different rankings show that large misses affect the models differently.

The median baseline has RMSE of $88,699. All fitted alternatives improve on it, though several have substantial training/validation gaps. Linear Regression's unstable CV result also limits confidence in its small held-out lead.

The next check is to inspect the largest residuals and compare raw versus engineered features within training CV. Ground-truth actuals and inputs are saved separately under `house_prices_dataset/ground_truth/`; repeated tuning against these rows would make them less useful as a final check.


In [ ]:
print("Best observed by ground-truth RMSE:", ", ".join(metadata["best_observed_models"]))
display(comparison[["model", "rmse", "mae", "r2", "best_model_flag", "underperforming_flag"]])
display(combined)
stage_comparison_figure(combined).show()
stage_comparison_figure(combined, "r2").show()
for name in ["training_performance", "testing_evaluation", "model_comparison"]:
    print("Report:", REPORTS_DIR / f"{name}.html")
    display(HTML(f'<a href="../reports/{name}.html" target="_blank">{name}</a>'))
